In [56]:
import yfinance as yf

Ticker = "NQ=F"

df = yf.download(Ticker, period="max", interval="15m")

df.columns.names = [None, None]

df.columns = df.columns.get_level_values(0)

df = df.drop(columns=["Volume"])

[*********************100%***********************]  1 of 1 completed


In [57]:
# import pandas as pd

# df = pd.read_csv("data.csv")

# df.set_index("timestamp ET", inplace=True)

# df.index.name = "Datetime"

# df = df[["Open", "High", "Low", "Close"]]



# df.index = pd.to_datetime(
#     df.index,
#     format="%m/%d/%Y %H:%M"
# ).tz_localize("America/New_York")

In [58]:
# df["Side"] = "None"

# df = df.tail(500) # for testing purposes

df

,Close,High,Low,Open
Datetime,,,,
2026-06-09 10:45:00-04:00,29239.75,29241.25,29133.25,29201.00
2026-06-09 11:00:00-04:00,29243.50,29302.00,29143.50,29242.00
2026-06-09 11:15:00-04:00,28859.25,29254.50,28848.50,29243.75
2026-06-09 11:30:00-04:00,28902.75,28944.25,28809.75,28861.25
2026-06-09 11:45:00-04:00,28695.25,28972.00,28681.50,28903.75
...,...,...,...,...
2026-08-07 15:45:00-04:00,29834.25,29860.25,29778.00,29778.75
2026-08-07 16:00:00-04:00,29820.25,29850.75,29805.25,29833.75
2026-08-07 16:15:00-04:00,29805.00,29829.00,29803.75,29820.50


In [59]:
RRR = 2

### Strategy

In [60]:
import numpy as np 

high = df["High"].to_numpy()
low = df["Low"].to_numpy()

outcome = np.full(len(df), 0.0)

MAX_LOOKAHEAD = 3*60

for root in range(len(df)):
    risk = high[root] - low[root]

    is_long = False
    is_short = False

    counter = 0

    for i in range(root + 1, len(df)):
        if counter >= MAX_LOOKAHEAD:break

        if not is_long and not is_short:

            if high[i] >= high[root] and low[i] > low[root]:
                is_long = True

            elif low[i] <= low[root] and high[i] < high[root]:
                is_short = True

            elif low[i] > low[root] and high[i] < high[root]: continue

            else: break  # Trade triggered and SL hit right after it

        if is_long:
            if low[i] <= low[root]: break # SL hit

            if high[i] - high[root] > RRR * risk: # TP hit
                outcome[root] = RRR
                break

        elif is_short:
            if high[i] >= high[root]: break # SL hit

            if low[root] - low[i] > RRR * risk: # TP hit
                outcome[root] = RRR
                break

In [61]:
df["Outcome"] = outcome

result_df = (
    df.groupby(df.index.strftime("%H:%M"))["Outcome"]
      .apply(lambda x: (x == RRR).mean())
      .rename("Win Rate")
      .to_frame()
)

result_df.index.name = "Time"

result_df["Expectancy"] = (RRR + 1)* result_df["Win Rate"] - 1
result_df["Win Rate"] = result_df["Win Rate"] * 100
result_df = result_df.sort_values("Win Rate", ascending=False)

result_df.head(20)

,Win Rate,Expectancy
Time,,
21:15,53.488372,0.604651
21:00,51.162791,0.534884
21:30,51.162791,0.534884
00:45,48.780488,0.463415
02:00,48.780488,0.463415
11:30,45.238095,0.357143
01:00,43.902439,0.317073
07:15,43.902439,0.317073
01:15,43.902439,0.317073


In [62]:
from datetime import datetime
from zoneinfo import ZoneInfo

dt = datetime.fromisoformat("2026-07-31 17:00:00-04:00")

morocco_time = dt.astimezone(ZoneInfo("Africa/Casablanca"))

print(morocco_time)

2026-07-31 22:00:00+01:00
